# CS 521 HW3, Problem 2: SynCode

Runtime > Change runtime type > **T4 GPU**. The notebook is self-contained (no external files).

`run_syncode(textual_query)` uses SynCode with Microsoft's Phi-2 to extract the two operands and the operator
from the query (the output is constrained by a Lark grammar), then calculates the answer in Python.
It returns `(operands_list, operator_str, answer_float)`.

In [1]:
# syncode 0.4.16 declares Python < 3.13, so on a newer Colab Python pip falls back to an old release whose
# pinned tokenizers has to be built from source (and fails). Install it without its pins, then its dependencies.
!pip install -q --no-deps --ignore-requires-python syncode==0.4.16
!pip install -q interegular fire jsonschema datasets "transformers==4.53.2"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
syncode 0.4.16 requires regex==2024.11.6, but you have regex 2025.11.3 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import re
from decimal import Decimal

import torch
from syncode import Syncode

MODEL_NAME = "microsoft/phi-2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Grammar

The output must be exactly `Operands: <number>, <number>` and `Operator: <op>` on two lines.
A number may have a sign and a decimal point (`327.`, `11.0`, `.5`). After the operator the grammar is complete,
so SynCode only allows the end-of-sequence token and the generation stops.

In [3]:
GRAMMAR = r"""
start: "Operands: " NUMBER ", " NUMBER "\nOperator: " OPERATOR
NUMBER: /-?([0-9]{1,15}(\.[0-9]{0,15})?|\.[0-9]{1,15})/
OPERATOR: "+" | "-" | "*" | "/"
"""

## Few-shot prompt

Four examples that show only the extraction (no final answers). The same text is put before every user query.

In [4]:
FEW_SHOT = """Extract the two numbers (in the order of the operation) and the operator (+, -, * or /) from the question.

Question: What is 12.5 times 4.0?
Operands: 12.5, 4.0
Operator: *

Question: What is 10.5 subtracted from 89.25?
Operands: 89.25, 10.5
Operator: -

Question: What is 7. divided by 2.0?
Operands: 7.0, 2.0
Operator: /

Question: What is the sum of 3.75 and 106.2?
Operands: 3.75, 106.2
Operator: +

Question: """

In [5]:
syn_llm = Syncode(
    model=MODEL_NAME,
    grammar=GRAMMAR,
    mode="grammar_strict",
    parse_output_only=True,  # the grammar constrains only the generated text, not the prompt
    device=DEVICE,
    do_sample=False,  # greedy decoding
    max_new_tokens=40,
)

[2026-10-28 06:37:28,768-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

[2026-10-28 06:39:00,362-syncode.mask_store.mask_store] - Using cache: True and fsm path cache/mask_stores/CodeGenTokenizerFast/grammar_strict_3098760452_50257.pkl exist: False
[2026-10-28 06:39:00,363-syncode.mask_store.mask_store] - Creating mask store for CodeGenTokenizerFast and custom, may take more than 10 minutes. Caching at /content/cache/mask_stores/CodeGenTokenizerFast/grammar_strict_3098760452_50257.pkl.
[2026-10-28 06:39:00,917-syncode.mask_store.fsm_set] - 5 FSMs with 80 states initialized in 0.01 seconds
[2026-10-28 06:39:00,917-syncode.mask_store.mask_store] - Ignore whitespace is False
[2026-10-28 06:39:00,930-syncode.mask_store.mask_store] - Number of 2 length terminal sequences reduced from 25 to 5


100%|██████████| 80/80 [00:23<00:00,  3.35it/s]

[2026-10-28 06:39:24,884-syncode.mask_store.mask_store] - Time taken to create mask store: 24.51 seconds


In [6]:
OUTPUT_RE = re.compile(r"\s*Operands:\s*(\S+?),\s*(\S+)\s*Operator:\s*([-+*/])")


def calculate(a, b, op):
    # +, - and * are exact in decimal arithmetic (45.1 + 23.54 = 68.64, not 68.64000000000001).
    da, db = Decimal(a), Decimal(b)
    if op == "+":
        return float(da + db)
    if op == "-":
        return float(da - db)
    if op == "*":
        return float(da * db)
    return float(a) / float(b) if float(b) != 0 else float("nan")


def run_syncode(textual_query):
    prompt = FEW_SHOT + textual_query.strip() + "\n"
    output = syn_llm.infer(prompt)[0]
    m = OUTPUT_RE.match(output)
    if m is None:
        raise ValueError(f"could not parse the SynCode output: {output!r}")
    a, b, op = m.groups()
    return [float(a), float(b)], op, calculate(a, b, op)

In [7]:
tests = [
    ("What is 327. multiplied by 11.0?", 3597.0),
    ("What is 45.1 plus 23.54?", 68.64),
    ("What is 120.4 divided by 4.0?", 30.1),
    ("What is 1000.5 minus 0.75?", 999.75),
    ("What is 2.5 times 2.5?", 6.25),
    ("What is 0.1 plus 0.2?", 0.3),
    ("What is 99.0 divided by 3.?", 33.0),
    ("What is 15.25 subtracted from 40.0?", 24.75),
]
passed = 0
for q, expected in tests:
    result = run_syncode(q)
    ok = abs(result[2] - expected) < 1e-9
    passed += ok
    print(f"{'PASS' if ok else 'FAIL'}  {q:<40} -> {result}  (expected {expected})")
print(f"{passed}/{len(tests)} passed")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


PASS  What is 327. multiplied by 11.0?         -> ([327.0, 11.0], '*', 3597.0)  (expected 3597.0)
PASS  What is 45.1 plus 23.54?                 -> ([45.1, 23.54], '+', 68.64)  (expected 68.64)
PASS  What is 120.4 divided by 4.0?            -> ([120.4, 4.0], '/', 30.1)  (expected 30.1)
PASS  What is 1000.5 minus 0.75?               -> ([1000.5, 0.75], '-', 999.75)  (expected 999.75)
PASS  What is 2.5 times 2.5?                   -> ([2.5, 2.5], '*', 6.25)  (expected 6.25)
PASS  What is 0.1 plus 0.2?                    -> ([0.1, 0.2], '+', 0.3)  (expected 0.3)
PASS  What is 99.0 divided by 3.?              -> ([99.0, 3.0], '/', 33.0)  (expected 33.0)
PASS  What is 15.25 subtracted from 40.0?      -> ([40.0, 15.25], '-', 24.75)  (expected 24.75)
8/8 passed
